# Web-Gold-40K — recovery v2.1 correction

This notebook preserves v2 and changes only factors justified by its first two epochs: exact needs-recovery balancing, raw conditional-strategy evaluation, a bbox-specific adapter, and direct gradient/update diagnostics. Run `smoke`, then `diagnostic` (one epoch), and run the five-epoch `mini` only if the diagnostics are healthy.

In [1]:
# 1. Pull the latest modular code and record the environment.
from pathlib import Path
import importlib.metadata as metadata
import json, os, subprocess, sys
REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
requirements = ['transformers>=4.49,<5', 'peft>=0.14,<1', 'bitsandbytes>=0.45,<1', 'accelerate>=1,<2', 'scikit-learn>=1.4,<2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements], check=True)
for module_name in list(sys.modules):
    if module_name == 'web_agent' or module_name.startswith('web_agent.'):
        del sys.modules[module_name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(REPO_ROOT)
environment = {name: metadata.version(name) for name in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn']}
environment['python'] = sys.version.split()[0]
environment['git_commit'] = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
Path('/kaggle/working/gold_recovery_v2_1_environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))

Cloning into '/kaggle/working/webagent'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.4 MB/s eta 0:00:00
{
  "torch": "2.10.0+cu128",
  "transformers": "4.57.6",
  "peft": "0.19.1",
  "bitsandbytes": "0.49.2",
  "accelerate": "1.14.0",
  "scikit-learn": "1.9.0",
  "python": "3.12.13",
  "git_commit": "049588fbcbf5bd4be58fadee5715436e0c2eebf0"
}


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [2]:
# 2. Locate the attached dataset without downloading or extracting it.
ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SPLIT_FILES = ('split_train.json', 'split_val.json', 'split_test.json')
def find_split_root(root: Path) -> Path:
    if all((root / name).is_file() for name in SPLIT_FILES):
        return root
    candidates = []
    for current, _, files in os.walk(root, followlinks=True):
        if set(SPLIT_FILES).issubset(files):
            candidates.append(Path(current))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one structured split folder; found {candidates}')
    return candidates[0]
DATA_ROOT = find_split_root(ATTACHED_ROOT).resolve()
print('DATA_ROOT =', DATA_ROOT)

DATA_ROOT = /kaggle/input/datasets/kiyasmahmud/web-gold-40k/final_data_set_40k


In [3]:
# 3. Select one gate. Always run smoke before diagnostic; mini comes last.
import torch
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
STAGE = 'diagnostic'  # 'smoke', 'diagnostic', or 'mini'
SEED = 42
TRAIN_ROWS, VAL_ROWS = 5_000, 500
EPOCHS = 1 if STAGE == 'diagnostic' else 5
assert STAGE in {'smoke', 'diagnostic', 'mini'}
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
set_seed(SEED)
cfg = load_config('configs/backbones/qwen2vl_2b_gold_v2_1.yaml')
cfg['data']['root'] = str(DATA_ROOT)
cfg['data']['num_workers'] = 0 if STAGE == 'smoke' else 4
assert cfg['model']['task_adapters']['separate_bbox']
assert cfg['loss']['needs_recovery_weight_scheme'] == 'exact_inverse_frequency'
print('GPU:', torch.cuda.get_device_name(0), '| stage:', STAGE, '| epochs:', EPOCHS)

GPU: Tesla T4 | stage: diagnostic | epochs: 1


In [4]:
# 4. Run the selected gate. No training stage opens split_test.json.
from web_agent.train.gold_stages import build_processor, run_gold_mini, run_gold_smoke
from web_agent.utils.results import save_mini_diagnostics_json, save_mini_result_csv
processor = build_processor(cfg)
if STAGE == 'smoke':
    stage_report = run_gold_smoke(cfg, processor=processor, rows=16, seed=SEED)
else:
    stage_report = run_gold_mini(cfg, processor=processor, train_rows=TRAIN_ROWS, val_rows=VAL_ROWS, epochs=EPOCHS, seed=SEED)
report_path = Path(f'/kaggle/working/gold_recovery_v2_1_{STAGE}_report.json')
report_path.write_text(json.dumps(stage_report, indent=2), encoding='utf-8')
result_csv_path = diagnostics_path = None
if STAGE != 'smoke':
    result_csv_path = save_mini_result_csv(stage_report, f'/kaggle/working/gold_recovery_v2_1_{STAGE}_result.csv')
    diagnostics_path = save_mini_diagnostics_json(stage_report, f'/kaggle/working/gold_recovery_v2_1_{STAGE}_diagnostics.json')
print(json.dumps(stage_report, indent=2))
print('Report:', report_path, '| CSV:', result_csv_path, '| diagnostics:', diagnostics_path)

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/429M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290
train rows: 5000
action weights: [1.01, 1.01, 1.02, 1.01, 0.88, 1.1]
failure weights: [0.57, 0.97, 0.9, 8.33]
outcome weights: [1.3, 1.0]
recovery weights: [0.0, 0.0, 0.83, 1.49, 0.67, 0.0]
recovery pos_weight: 1.93 (attempted success=407, failure=784)
needs-recovery pos_weight: 3.2 (needed=1191, not-needed=3809)
epoch 0 | step 50/157 | loss 0.684 | elapsed 42.2min | ETA 90.2min
epoch 0 | step 100/157 | loss 0.326 | elapsed 84.0min | ETA 47.9min
epoch 0 | step 150/157 | loss 0.253 | elapsed 125.9min | ETA 5.9min
epoch 0: train_loss=0.5809  failure_f1=0.7636  failure_macro_f1=0.7374  outcome_bal_acc=0.7442  outcome_mcc=0.4817  success_recall=0.7692  outcome_brier=0.1686  outcome_majority_acc=0.5840  outcome_majority_macro_f1=0.3687  failtype_acc=0.4620  failtype_macro_f1=0.3942  failtype_bal_acc=0.4198  failtype_mcc=0.2925  failtype_majority_acc=0.4160  failtype_majority_macro_f1=0.1469  action_acc=0.3420  a

In [5]:
# 5. Enforce engineering evidence before spending another five epochs.
assert stage_report['status'] == 'PASS'
if STAGE == 'smoke':
    for name in ('bbox', 'needs_recovery', 'strategy', 'recovery_outcome', 'grounding_adapter'):
        assert stage_report['probe_gradient_norms'][name] > 0, f'No gradient: {name}'
        assert stage_report['probe_update_norms'][name] > 0, f'No update: {name}'
    assert stage_report['bbox_supervised_rows'] > 0
    assert min(stage_report['spatial_tokens_per_row']) > 0
    print('SMOKE PASSED. Restart, set STAGE to diagnostic, then Run All.')
else:
    assert stage_report['test_rows_read'] == 0
    assert len(stage_report['history']) == EPOCHS
    assert stage_report['checkpoint_roundtrip']
    assert result_csv_path.is_file() and diagnostics_path.is_file()
    last = stage_report['history'][-1]
    print('needs pos_weight:', stage_report['class_weights']['needs_recovery_pos_weight'])
    print('needs recovery:', last['needs_recovery_macro_f1'], last['needs_recovery_mcc'], 'classes:', last['needs_recovery_pred_classes'])
    print('raw attempted strategy:', last['strategy_attempted_macro_f1'], last['strategy_attempted_pred_classes'])
    print('bbox:', last['bbox_mean_iou'], last['bbox_recall_iou50'])
    print('bbox gradient/update:', last['train_grad_bbox_first'], last['train_update_bbox_norm'])
    print('bbox uncertainty multiplier:', last['train_uncertainty_multiplier_bbox'])
    if STAGE == 'diagnostic':
        print('Inspect these values before authorizing the five-epoch mini run.')
    else:
        assert stage_report['loss_decreased'] is True
        assert len(stage_report['epoch_checkpoints']) == EPOCHS
        print('FIVE-EPOCH ENGINEERING PASS. Decide quality from the registered gates.')

needs pos_weight: 3.198152780532837
needs recovery: 0.6125927738357393 0.227419265471854 classes: 2
raw attempted strategy: 0.3350485991995426 2
bbox: 0.007923804616834082 0.0
bbox gradient/update: 0.061085596680641174 0.221417635679245
bbox uncertainty multiplier: 1.0821511676184112
Inspect these values before authorizing the five-epoch mini run.


## Diagnostic decision

Do not start `mini` unless the smoke proves nonzero gradients and updates for every named probe. After `diagnostic`, verify that needs-recovery predicts both classes, raw attempted-strategy metrics are above their majority baseline, and bbox predictions/IoU moved. A completed notebook is engineering evidence—not a 90% result.